In [1]:
import requests
import pandas as pd

In [6]:
EIA_API_KEY = 'nKn8mDZ0iVhfdW8ZqM96chUC6CBSarxW0fUnfYYf' 

START_DATE = '2020-01-01'

SERIES_BRENT = 'RBRTE'
SERIES_WTI = 'RWTC'   

In [7]:
def fetch_eia_prices(series_id, price_name):
    """Fetches daily spot prices for a given EIA series ID."""
    print(f"Fetching {price_name} prices from {START_DATE}...")
    
    url = (
        f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
        f"?api_key={EIA_API_KEY}"
        f"&frequency=daily"
        f"&data[0]=value"
        f"&facets[series][]={series_id}"
        f"&start={START_DATE}"
        f"&sort[0][column]=period"
        f"&sort[0][direction]=desc"
        f"&length=5000"
    )
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()['response']['data']
        df = pd.DataFrame(data)
        
        # Clean and isolate the columns
        df = df[['period', 'value']]
        df.rename(columns={'period': 'Date', 'value': price_name}, inplace=True)
        
        # Convert to datetime and strip timezone
        df['Date'] = pd.to_datetime(df['Date'])
        if df['Date'].dt.tz is not None:
            df['Date'] = df['Date'].dt.tz_localize(None)
            
        return df
    else:
        print(f"Failed to fetch {price_name}. Status: {response.status_code}")
        return pd.DataFrame() # Return empty df on failure

In [ ]:
df_brent = fetch_eia_prices(SERIES_BRENT, 'Brent_Price_USD')
df_wti = fetch_eia_prices(SERIES_WTI, 'WTI_Price_USD')

# 2. Merge them together on the Date column
if not df_brent.empty and not df_wti.empty:
    print("Merging Brent and WTI datasets...")
    # 'inner' join ensures we only keep days where both markets were actively trading
    df_combined = pd.merge(df_brent, df_wti, on='Date', how='inner')
    
    # Sort chronologically
    df_combined.sort_values(by='Date', inplace=True)
    
    # Calculate the spread (price difference) just in case you want to visualize it
    df_combined['Brent_WTI_Spread'] = df_combined['Brent_Price_USD'] - df_combined['WTI_Price_USD']
    
    # 3. Export to CSV
    output_file = 'data/Brent_WTI_Prices_2020_to_Present.csv'
    df_combined.to_csv(output_file, index=False)


Fetching Brent_Price_USD prices from 2020-01-01...
